In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [8]:
data = pd.read_csv("Dataset/Dataset.csv", lineterminator='\n')
data = data.replace("Not Given", np.nan)

data["date_added"] = pd.to_datetime(data["date_added"], format = "%m/%d/%Y", errors = "coerce") #convert data_added column to standard datetime format
data["year_added"] = data["date_added"].dt.year #Extract only the years
data["duration_no"] = data["duration"].str.split(" ").str[0].astype(float) #Split the duration column into a number and unit
data["duration_unit"] = data["duration"].str.split(" ").str[1] # "90 min" becomes 90 and "min" becomes unit 

data = data.drop_duplicates(subset=["title", "type", "release_year"]) # Removing duplicates movies/TVshows

print("Missing Data:\n" ,data.isna().sum())
print("\nMovie vs TV show split:\n", data["type"].value_counts(normalize = True))


Missing Data:
 show_id             0
type                0
title               0
director         2587
country           287
date_added          0
release_year        0
rating              0
duration            0
listed_in\r         0
year_added          0
duration_no         0
duration_unit       0
dtype: int64

Movie vs TV show split:
 type
Movie      0.696939
TV Show    0.303061
Name: proportion, dtype: float64


In [9]:
categorized = ["rating"]
numeric_features = ["release_year", "year_added"]
X = data[categorized + numeric_features]
y = data["type"]

preprocess = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", Pipeline (steps =[
            ("imputer", SimpleImputer(strategy="constant", fill_value="Unknown")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), categorized)]
)

pipeline = Pipeline(steps=[
    ("preprocess", preprocess),
    ("classify", RandomForestClassifier(n_estimators = 100, random_state= 42, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"Pipeline Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"\nDetailed Report\n: {classification_report(y_test, y_pred)}")


Pipeline Accuracy: 0.6695

Detailed Report
:               precision    recall  f1-score   support

       Movie       0.84      0.65      0.73      1225
     TV Show       0.47      0.71      0.57       533

    accuracy                           0.67      1758
   macro avg       0.66      0.68      0.65      1758
weighted avg       0.73      0.67      0.68      1758



In [10]:
data.head()

,show_id,type,title,director,country,date_added,release_year,rating,duration,listed_in\r,year_added,duration_no,duration_unit
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,United States,2021-09-25,2020,PG-13,90 min,Documentaries\r,2021,90.0,min
1,s3,TV Show,Ganglands,Julien Leclercq,France,2021-09-24,2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",2021,1.0,Season
2,s6,TV Show,Midnight Mass,Mike Flanagan,United States,2021-09-24,2021,TV-MA,1 Season,"TV Dramas, TV Horror, TV Mysteries\r",2021,1.0,Season
3,s14,Movie,Confessions of an Invisible Girl,Bruno Garotti,Brazil,2021-09-22,2021,TV-PG,91 min,"Children & Family Movies, Comedies\r",2021,91.0,min
4,s8,Movie,Sankofa,Haile Gerima,United States,2021-09-24,1993,TV-MA,125 min,"Dramas, Independent Movies, International Movi...",2021,125.0,min
